# Week 3 — Monday: Cleaning & Grouping Data

**DATA 202 · Calvin University**

> Hagar, Sarah's Egyptian servant, flees into the desert after mistreatment. There, an angel of the Lord finds her by a spring and speaks to her. She responds:
>
> *"You are a God of seeing,"* she said, *"for truly here I have seen him who looks after me."* — Genesis 16:13

**Today's theme:** cleaning & grouping data = a form of *seeing* people who are easy to overlook (Stoddart's "com-veillance")

**Dataset:** people experiencing homelessness — deliberately messy

**Plan (~50 min):**

| Time | Section |
|---|---|
| ~5 min | Load & inspect the messy data |
| ~22 min | Part 1 — Cleaning strings with regex (SLO 03A) |
| ~18 min | Part 2 — Grouping & aggregating (SLO 03B) |
| ~5 min | Careful with aggregations + what's next |

**Watch for:**
- 🎯 **Predict First** — guess before we run the code
- 🙋 **Quick Check** — verbal, no code, just think

---
## Loading the Data

In [ ]:
import pandas as pd

DATA_PATH = "../../datasets/homeless.csv"
homeless = pd.read_csv(DATA_PATH)
homeless.head()

In [ ]:
homeless.info()

**What to notice:**
- names: stray spaces — `"   peter"`
- city: same place, 5+ spellings — `NEW-YORK`, `new-york`, `New york`, `SF`, `L.A.`...
- `shelter_status`: mixed case & phrasing — `SHELTERED`, `shelter , pending`

→ `groupby("city")` right now = a dozen "different" cities that are really 5

🙋 **Quick Check:** Skim the columns above — how many *actual* distinct cities are hiding in this data? Say a number out loud.

---
## Part 1: Cleaning String Data (SLO 03A) · ~22 min

**Tools:** pandas string ops (`.str.___`) + regular expressions (regex)

### Regex: Pattern Matching for Text

- regex = a *pattern* language, not exact text — "find text that looks like this"

| Syntax | Meaning | Example | Matches |
|---|---|---|---|
| literal text | the exact characters | `cat` | `cat` |
| `\|` | OR (alternation) | `cat\|dog` | `cat` or `dog` |
| `[A-Z]` | a character class | `[A-Z]+` | one or more uppercase letters |
| `[^...]` | NOT this class | `[^a-zA-Z\s]` | anything that isn't a letter or space |
| `\d` | a digit | `\d{3}` | exactly 3 digits, e.g. `422` |
| `\s` | whitespace | `\s+` | one or more spaces/tabs |
| `.` | any single character | `h.t` | `hat`, `hot`, `h5t`... |
| `*` `+` `?` | quantifiers | `go+gle` | `gogle`, `google`, `gooogle`... |
| `^` ... `$` | start ... end of string | `^shelter$` | the *whole* string is exactly `shelter` |
| `(...)`  | a group, often with `\|` inside | `(shelter\|street)` | `shelter` or `street`, treated as one unit |

Practice at [regexone.com](https://regexone.com).

🎯 **Predict First:** match or no match? (yes/no)

1. `shelter.*pending` vs `"shelter , pending"`
2. `^sheltered$` vs `"sheltered"`
3. `^sheltered$` vs `"unsheltered"`
4. `^sheltered$` vs `"Sheltered"`

Guess, *then* run the cell below.

In [ ]:
import re

tests = [
    (r"shelter.*pending", "shelter , pending"),
    (r"^sheltered$", "sheltered"),
    (r"^sheltered$", "unsheltered"),
    (r"^sheltered$", "Sheltered"),
]
for pattern, text in tests:
    print(f"{pattern!r:22} vs {text!r:22} -> {bool(re.search(pattern, text))}")

**#3 & #4 are both `False`:**
- `^...$` = must match the **entire** string → `unsheltered` fails
- regex is **case-sensitive by default** → `Sheltered` ≠ `sheltered`
- → we'll lowercase text *before* matching, coming up

---
### 🔨 Mini-Task A — Write a Regex (~2 min)

Match strings of **only letters and spaces** — nothing else. Test with `re.fullmatch()`.

*Hint:* character class + `+`, anchored to the whole string.

In [ ]:
# Your code here
my_pattern = r""  # fill this in

for s in ["New York", "New York3", "New-York"]:
    print(s, "->", bool(re.fullmatch(my_pattern, s)))

---
### Now let's clean for real

In [ ]:
# 1. Strip stray whitespace
homeless["name"] = homeless["name"].str.strip()
homeless["city"] = homeless["city"].str.strip()
homeless[["name", "city"]].head()

🎯 **Predict First:** we're about to (a) turn dashes → spaces, (b) strip non-letter/space chars, (c) title-case. What does `"L.A."` become? `"new-york"`? Guess first.

In [ ]:
# 2. Standardize separators, drop stray punctuation, then title-case
homeless["name"] = (
    homeless["name"]
    .str.replace(r"[^a-zA-Z\s-]", "", regex=True)   # "chloe." -> "chloe"
    .str.title()                                     # -> "Chloe"
)

homeless["city"] = (
    homeless["city"]
    .str.replace("-", " ", regex=False)               # "new-york" -> "new york"
    .str.replace(r"[^a-zA-Z\s]", "", regex=True)      # "CHICAGO." -> "CHICAGO"
    .str.title()                                       # -> "Chicago"
)

homeless[["name", "city"]].drop_duplicates(subset="city").sort_values("city")

**Still abbreviated:** `Sf`, `La` — title-casing an abbreviation ≠ a full name
- regex/case rules only go so far → sometimes you need an explicit **lookup**

🙋 **Quick Check:** why *can't* regex fix `"SF"` → `"San Francisco"`, when it fixed `"CHICAGO."` → `"Chicago"` just fine? What's fundamentally different?

In [ ]:
# 3. Map the remaining abbreviations explicitly
city_map = {"Sf": "San Francisco", "La": "Los Angeles"}
homeless["city"] = homeless["city"].replace(city_map)
homeless["city"].value_counts()

### Two regex, same job

- rarely only one correct pattern
- both lines below flag the same rows — why, before reading on?

In [ ]:
sample = pd.Series(["shelter , pending", "shelter,pending", "shelter  ,  pending", "sheltered"])

version_a = sample.str.contains(r"shelter\s*,\s*pending", regex=True)
version_b = sample.str.contains(r"shelter[ ]*,[ ]*pending", regex=True)

pd.DataFrame({"text": sample, "version_a": version_a, "version_b": version_b})

- `\s*` vs `[ ]*`: both ≈ zero-or-more spaces here
- `\s` also matches tabs/newlines; `[ ]` = literal space only
- small choices like this matter once data gets messier than expected

In [ ]:
# 4. Standardize shelter_status the same way — case first, then targeted replacements
homeless["shelter_status"] = homeless["shelter_status"].str.strip().str.lower()

homeless["shelter_status"] = (
    homeless["shelter_status"]
    .str.replace(r"^sheltered$", "shelter", regex=True)              # "sheltered" -> "shelter"
    .str.replace(r"shelter\s*,\s*pending", "shelter pending", regex=True)
    .str.replace(r"temporary shelter", "shelter temporary", regex=True)
)
homeless["shelter_status"].value_counts()

---
### 🔨 Mini-Task B — Extend the Pattern (~3 min)

Some rows spell it `"temp shelter"` instead of `"temporary shelter"`.

- Write **one** pattern (`|` alternation) that matches **either** spelling
- One `.str.replace()` call, test below

In [ ]:
# Your code here
sample = pd.Series(["temporary shelter", "temp shelter", "shelter"])
my_pattern = r""  # fill this in — should match "temporary shelter" OR "temp shelter"

sample.str.replace(my_pattern, "shelter temporary", regex=True)

In [ ]:
# 5. education_level: strip + lowercase collapses most of the mess, then relabel nicely
homeless["education_level"] = (
    homeless["education_level"]
    .str.strip()
    .str.lower()
    .replace({"none": "None", "primary": "Primary", "secondary": "Secondary", "higher": "Higher"})
)
homeless["education_level"].value_counts()

---
### 🔨 Task 1 — Flag a Pattern in Free Text (~5 min)

`notes` = unstructured text, still has signal. Flag every row mentioning losing a job with `.str.contains()` + regex.

- *Hint:* phrasings vary — `"job loss during pandemic"`, `"lost JOB; looking for work"`. `r"job"` + `case=False` catches both.
- Assign to `homeless["job_related"]`
- **Bonus:** add `|` to also flag `"healthcare"`. How many rows match now?

In [ ]:
# Your code here


---
## Part 2: Grouping and Aggregating (SLO 03B) · ~18 min

Sometimes we want to **summarize groups**, not individual rows:

* How many people per city?
* Average support amount by shelter status?
* Which group has the highest average years homeless?

→ `groupby()` + aggregation. More than one way to ask each question.

In [ ]:
# 1. Grouping by one column: how many people per city?
homeless.groupby("city")["id"].count()

🎯 **Predict First:** before we look at money — which city has the **most** people? The **fewest**? Guess, then check the output above.

In [ ]:
# 2. Aggregating a numeric column: average monthly support by city
homeless.groupby("city")["monthly_support_usd"].mean()

### Many ways to summarize the same numbers

- `.mean()` = only one lens
- pass a **list** of function names to `.agg()` → runs several at once, side by side:

In [ ]:
homeless.groupby("shelter_status")["monthly_support_usd"].agg(["mean", "median", "min", "max", "std"])

* **mean** — the arithmetic average; sensitive to a few extreme values.
* **median** — the middle value; barely moves even if one entry is way off.
* **min / max** — the range of what's actually happening in each group.
* **std** — how spread out the values are; a small std means the group is fairly uniform.

🙋 **Quick Check:** if one person's `monthly_support_usd` were mistakenly entered as `50000` instead of `500`, which statistic above would be thrown off the most — the mean or the median? Which would barely notice?

In [ ]:
# 3. Grouping by multiple columns: average years homeless by city AND education level
homeless.groupby(["city", "education_level"])["years_homeless"].mean()

### Naming your aggregations

- dict-style `.agg({...})` → good for summarizing several *columns*
- want several *statistics* from the same column, clean names? → **named aggregation**

In [ ]:
# Dict-of-lists style: several statistics on several columns
homeless.groupby("shelter_status").agg({
    "family_size": ["mean", "max"],
    "monthly_support_usd": ["mean", "sum"],
})

In [ ]:
# Named-aggregation style: same idea, flatter and more readable output
homeless.groupby("city").agg(
    avg_support=("monthly_support_usd", "mean"),
    max_support=("monthly_support_usd", "max"),
    n_people=("id", "count"),
)

Same work, different syntax — pick whichever reads more clearly for the summary you're building.

---
### 🔨 Mini-Task C — Same Aggregation, Other Syntax (~3 min)

Rewrite **average & max `years_homeless` per `education_level`** using named aggregation (`.agg(name=(...))`) instead of dict style.

In [ ]:
# For reference, dict style:
homeless.groupby("education_level").agg({"years_homeless": ["mean", "max"]})

# Your code here — same result, named-aggregation style


### Beyond the built-ins: custom aggregations

- `.agg()` takes **any function**, including a `lambda`
- e.g. the *range* (max − min) of support per city:

In [ ]:
homeless.groupby("city")["monthly_support_usd"].agg(lambda x: x.max() - x.min())

- groupby results carry the grouping column as an **index**
- `.reset_index()` → back to a normal column — useful before sorting, plotting, or merging

🎯 **Predict First:** which city gets the highest **total** `monthly_support_usd`? Same city as highest **average**? Guess both, then check.

In [ ]:
# reset_index(), then sort to find the largest group
(
    homeless.groupby("city")["monthly_support_usd"]
    .sum()
    .reset_index()
    .sort_values("monthly_support_usd", ascending=False)
)

In [ ]:
# value_counts() is a shortcut for "groupby + count" on one column
homeless["education_level"].value_counts()

---
### 🔨 Task 2 — Group, Aggregate, Compare (~5 min)

1. Which `shelter_status` has the highest **average** `monthly_support_usd`?
2. Among people with more than 10 years homeless (`years_homeless > 10`), which `city` has the most people?
3. Using **named aggregation**, compute both the **count** of people and the **average** `years_homeless`, per `education_level`, in a single `.agg(...)` call.

*Hint for (2): filter first, then group.*

In [ ]:
# Your code here


---
## Careful with Aggregations

Every aggregation **distorts** the data on purpose:

* **Counts** — *how many*, not *who*
* **Means** — smooth over differences (7-yr avg could mean "everyone near 7" or "half brand-new, half decade+")
* **Medians** — resist outliers, but hide them too
* **Sums** — favor big groups (highest *total* ≠ highest *per person*)

**Big idea:** aggregation trades detail for a visible pattern — not a flaw, the whole point.

→ **Fri Forum 1**, *Counting* Ch. 1 (Deborah Stone): no summary number is "raw" — someone decided what counts as alike before any counting began. `groupby()` *is* that decision, made in code.

---
## Coming Up

| Day | Topic | Builds on today |
|---|---|---|
| Wed | Choosing the right plot | The same cleaned dataset, now visualized — histograms, scatter, line, and bar charts |
| Fri | Forum 1 — *Counting*, Ch. 1 | What gets ignored when `groupby()` treats rows as "the same"? |
| Week 4 | Joining tables | Combining datasets *before* you can group or plot them together |